# Task 1: Data Preparation and QA Pair Generation

In this notebook, I will walk through the data preparation process and generate QA (Question-Answer) pairs for my Retrieval-Augmented Generation (RAG) system.

Here is what I'll be doing step-by-step:
1. **Downloading the Data**: I'll download Chapter 10 of the Speech and Language Processing textbook.
2. **Text Cleaning**: I need to clean up the extracted text to remove noisy artifacts like page numbers and headers.
3. **Chunking**: I will chunk the text into manageable pieces for better retrieval later.
4. **API Setup**: I'll prompt for my Gemini API key securely so I don't accidentally leak it in the code.
5. **QA Generation**: Finally, I'll use the new Gemini 3.0 Flash model to automatically generate 20 robust QA pairs strictly from my text chunks.

In [4]:
# First, I need to make sure all the necessary libraries are installed in my environment.
# I'm using the standard subprocess module to run pip install programmatically to avoid Jupyter magic command issues.
import sys
import subprocess

# Install essential packages: PyMuPDF for PDF loading, Langchain text splitters, and Google's GenAI tools.
subprocess.check_call([sys.executable, "-m", "pip", "install", "pymupdf", "langchain", "langchain-text-splitters", "langchain-google-genai", "langchain-community"])


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.5 MB/s  0:00:009.7 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 1.9 MB/s  0:00:001.2 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain-community]m7/8 [langchain-community]


0

In [5]:
# Now I'll import all the necessary modules for my workflow.
import os                  # For file and environment variable management
import re                  # Regular expressions for cleaning the text
import json                # For saving my generated QA pairs
import getpass             # Securely prompt for the API key

# LangChain components that I rely on for loading and processing the documents
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# LangChain Google GenAI wrapper and prompt templates for my QA generation task
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate


/opt/homebrew/Caskroom/miniforge/base/envs/ai_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load and Clean the PDF Text

Before I can chunk the text, I need to download the textbook and strip out noisy text like page numbers and chapter headers.
This ensures my RAG model won't get confused by irrelevant words dividing the text blocks.

In [6]:
# I'm defining a helper function here to consistently clean up each page of the PDF text.
def clean_text(text):
    # Get rid of multiple empty lines
    text = re.sub(r'\n+', '\n', text)
    # Filter out standalone page numbers sitting on their own line
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    # Remove the chapter headers and book title footers completely
    text = re.sub(r'CHAPTER \d+.*$', '', text, flags=re.MULTILINE)
    text = re.sub(r'Speech and Language Processing.*$', '', text, flags=re.MULTILINE)
    # Strip any leading/trailing whitespace
    return text.strip()

pdf_path = "10.pdf"

# If I haven't downloaded Chapter 10 yet, I'll grab it dynamically using urllib
if not os.path.exists(pdf_path):
    print("Downloading Chapter 10 PDF...")
    import urllib.request
    urllib.request.urlretrieve("https://web.stanford.edu/~jurafsky/slp3/10.pdf", pdf_path)
    
# I'm using LangChain's PyMuPDFLoader to extract the text content from the PDF locally
loader = PyMuPDFLoader(pdf_path)
documents = loader.load()

# Iterate through each page, clean it up, and concatenate it into a huge string
full_text = ""
for doc in documents:
    full_text += clean_text(doc.page_content) + "\n"

print(f"Total cleaned text length: {len(full_text)} characters.")


Total cleaned text length: 54915 characters.


In [7]:
# I need to break the large text down into smaller 'chunks' to eventually feed into my model.
# I chose a chunk size of 700 characters with an overlap of 100 characters to ensure context isn't lost between boundaries.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=100)

# Apply the splitter to my cleaned full text
chunks = text_splitter.create_documents([full_text])
print(f"I've successfully created {len(chunks)} chunks from the document.\n")

# Let's peek at the first chunk so I can verify the splitting worked correctly
print("--- Sample Chunk 1 ---")
print(chunks[0].page_content)


I've successfully created 100 chunks from the document.

--- Sample Chunk 1 ---
Daniel Jurafsky & James H. Martin.
Copyright © 2026.
All
rights reserved.
Draft of January 6, 2026.
CHAPTER


## 2. Setting Up the LLM

To generate my QA pairs, I'll use the new Gemini 3.0 Flash model. I don't want to hardcode my API key directly into the script, so I'll prompt for it dynamically instead. Secure handling of keys is always the best practice!

In [10]:
# Ask myself securely for the API key at runtime using the getpass module
api_key = getpass.getpass("Please enter your Gemini API Key: ")
os.environ["GEMINI_API_KEY"] = api_key

# Initialize the ChatGoogleGenerativeAI class with the latest gemini-2.5-flash model
# I'm setting temperature=0 to make the outputs deterministic so my QA pairs are strictly based on the provided facts.
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
print("LLM Initialized and ready to go!")


LLM Initialized and ready to go!


## 3. Generating Question-Answer Pairs

Now for the fun part! I'll define a Prompt Template that instructs Gemini to formulate a strict Question and Answer based entirely on a given chunk of text.

I want my output to be reliably formatted to use later, so I'm demanding a JSON object. I will select 20 equidistant chunks spanning the whole text and generate my independent QA pairs.

In [11]:
# I'm setting up my Prompt Template to firmly direct the LLM.
# It MUST act strictly on the chunk provided and MUST output raw JSON format.
prompt = PromptTemplate.from_template(
    "Generate a clear, standalone Question and Answer pair based STRICTLY on the following text chunk.\n"
    "Format your output exactly as a JSON object with 'question' and 'ground_truth_answer' keys.\n\n"
    "Text Chunk:\n{chunk}\n\n"
    "Output JSON only:"
)

# I'm building my logic chain by piping the prompt directly into the initialized LLM
qa_chain = prompt | llm

qa_pairs = []
# To get a good representation across the whole chapter, I'll sample 20 chunks evenly spread out.
step = len(chunks) // 20
selected_indices = [i * step for i in range(20)]

# Iterate over my selected snippet indices
for i, idx in enumerate(selected_indices):
    chunk_text = chunks[idx].page_content
    try:
        # Ask my chain to execute the prompt filled with my specific text chunk
        response = qa_chain.invoke({"chunk": chunk_text})
        
        # Sometimes LLMs sneak in formatting backticks, so I need to cautiously clean up the response before loading it as JSON
        response_text = response.content.strip()
        if response_text.startswith("```json"):
            response_text = response_text[7:]
        if response_text.endswith("```"):
            response_text = response_text[:-3]
            
        # Parse output into a Python dictionary
        qa_pair = json.loads(response_text)
        
        # Here I'm initializing some placeholder fields in my dictionary for the later indexing and retrieval experiments
        qa_pair["naive_rag_answer"] = ""
        qa_pair["contextual_retrieval_answer"] = ""
        
        # Add the completed dictionary to my master list
        qa_pairs.append(qa_pair)
        print(f"Successfully generated pair {i+1}/20.")
        
    except Exception as e:
        # Fallback if something goes wrong (e.g. malformed JSON or API disconnect)
        print(f"Uh oh, there was an error generating QA for chunk {idx}: {e}")


Successfully generated pair 1/20.
Successfully generated pair 2/20.
Successfully generated pair 3/20.
Successfully generated pair 4/20.
Successfully generated pair 5/20.
Successfully generated pair 6/20.
Successfully generated pair 7/20.
Successfully generated pair 8/20.
Successfully generated pair 9/20.
Successfully generated pair 10/20.
Successfully generated pair 11/20.
Successfully generated pair 12/20.
Successfully generated pair 13/20.
Successfully generated pair 14/20.
Successfully generated pair 15/20.
Successfully generated pair 16/20.
Successfully generated pair 17/20.
Successfully generated pair 18/20.
Successfully generated pair 19/20.
Successfully generated pair 20/20.


In [12]:
# Finally, I need to save my hard-earned QA pairs into a structured .json file.
output_path = "answer/response-st-126010-chapter-10.json"

# Make sure the target 'answer' directory exists before I try to write to it
os.makedirs("answer", exist_ok=True)

# Dump my list of dictionaries out nicely into the file 
with open(output_path, "w") as f:
    json.dump(qa_pairs, f, indent=2)
    
print(f"Done! I've successfully saved all {len(qa_pairs)} QA pairs out to {output_path}.")


Done! I've successfully saved all 20 QA pairs out to answer/response-st-126010-chapter-10.json.
